# Produce Breast Prediction CSVs

In [1]:
import sys
from pathlib import Path

start = Path.cwd().resolve()
for candidate in (start, *start.parents):
    sprint_dir = candidate / "codes" / "sprint"
    if (sprint_dir / "prediction_export.py").exists():
        PROJECT_ROOT = candidate
        if str(PROJECT_ROOT) not in sys.path:
            sys.path.insert(0, str(PROJECT_ROOT))
        break
else:
    raise RuntimeError(f"Cannot find codes/sprint/prediction_export.py from {start}")

LEGACY_DIR = PROJECT_ROOT / "codes" / "_legacy_models" / "breast"
if str(LEGACY_DIR) not in sys.path:
    sys.path.insert(0, str(LEGACY_DIR))

from codes.sprint.prediction_export import select_least_used_cuda_before_torch_import

select_least_used_cuda_before_torch_import()

Detected CUDA devices before torch import:
  physical=0 used=836 MB / 11264 MB (7.4%) <-- selected as cuda:0


In [2]:
import gc
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

from codes.sprint.prediction_export import export_model_specs, find_project_root

# Legacy model & dataset classes
from codes._legacy_models.breast.model import Model_C2, Model_RNA_Only
from codes._legacy_models.breast.utils import BreastMultimodalDataset, load_model_weights, set_seed

PROJECT_ROOT = find_project_root(Path.cwd())
print(f"Project root: {PROJECT_ROOT}")

set_seed(42)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Project root: /media/yuzhi/My PSSD/spatialProtein/spatialProtein/Github
Using device: cuda:0


In [3]:
# ---- Paths ----
TRAIN_H5AD = PROJECT_ROOT / "datas" / "breast" / "Breast_Human_Final.h5ad"
VAL_H5AD = PROJECT_ROOT / "datas" / "breast" / "Breast_Mouse_Final.h5ad"
MODEL_SAVE_ROOT = PROJECT_ROOT /"datas"/ "models" / "breast"
OUTPUT_DIR = PROJECT_ROOT /"datas"/ "outputs" / "breast"

MODEL_SPECS = [
    {"label": "C2", "model_class": Model_C2,
     "model_dir_candidates": ["C2"],
     "output_prefix": "breast_C2"}
]
BATCH_SIZE = 8

# ---- Image transform ----
common_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

for required_path in [TRAIN_H5AD, VAL_H5AD, MODEL_SAVE_ROOT]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
ad_train = sc.read_h5ad(TRAIN_H5AD, backed="r")
ad_val = sc.read_h5ad(VAL_H5AD, backed="r")
common_genes = sorted(
    set(g.upper() for g in ad_train.var_names).intersection(
        g.upper() for g in ad_val.var_names
    )
)
p_train = [str(x) for x in list(ad_train.uns.get("protein_names", []))]
p_val = [str(x) for x in list(ad_val.uns.get("protein_names", []))]
common_proteins = sorted(set(p_train).intersection(p_val))
if len(common_proteins) == 0 and p_train and p_val:
    p_train_upper = {p.upper(): p for p in p_train}
    p_val_upper = {p.upper(): p for p in p_val}
    common_proteins = [p_train_upper[u] for u in sorted(set(p_train_upper).intersection(p_val_upper))]
del ad_train, ad_val
gc.collect()

val_dataset = BreastMultimodalDataset(
    str(VAL_H5AD),
    target_genes=common_genes,
    target_proteins=common_proteins or None,
    transform=common_transform,
)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

NUM_GENES = val_dataset.rna_data.shape[1]
NUM_TARGETS = val_dataset.protein_data.shape[1]
target_names = common_proteins or [str(i) for i in range(NUM_TARGETS)]

print(f"Inference config: {NUM_GENES} genes, {NUM_TARGETS} proteins, "
      f"{len(val_dataset)} validation spots")
print(target_names)

Loading data from: Breast_Mouse_Final.h5ad ...
Inference config: 15462 genes, 11 proteins, 1978 validation spots
['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10']


In [5]:
saved_df, pred_df, target_df = export_model_specs(
    MODEL_SPECS,
    MODEL_SAVE_ROOT,
    OUTPUT_DIR,
    val_loader,
    device,
    NUM_TARGETS,
    NUM_GENES,
    target_names,
    load_model_weights,
)

display(saved_df)
display(pred_df.head())
display(target_df.head())

[C2] saved predictions: /media/yuzhi/My PSSD/spatialProtein/spatialProtein/Github/datas/outputs/breast/breast_C2_predictions.csv
[C2] saved targets:     /media/yuzhi/My PSSD/spatialProtein/spatialProtein/Github/datas/outputs/breast/breast_C2_targets.csv


,Model,Weights,ModelDir,Predictions,Targets,Rows,TargetsCount
0,C2,/media/yuzhi/My PSSD/spatialProtein/spatialPro...,C2,/media/yuzhi/My PSSD/spatialProtein/spatialPro...,/media/yuzhi/My PSSD/spatialProtein/spatialPro...,1978,11


,test_index,0,1,2,3,4,5,6,7,8,9,10
0,0,9.372547,9.096274,8.853979,8.002047,8.452836,8.839788,8.205502,10.146698,7.845893,9.698146,8.241592
1,1,9.577501,9.242154,9.169737,8.239794,8.643429,9.504675,8.505889,10.508209,8.111282,9.955764,8.551137
2,2,9.492790,9.703487,9.665708,8.623643,8.833637,10.414623,8.938528,10.668880,8.502625,10.209125,8.856269
3,3,10.420130,10.081185,9.756646,8.854205,9.353297,9.617031,9.020961,11.231889,8.650845,10.743786,9.084800
4,4,9.950808,9.664708,9.572763,8.606924,9.000311,9.929623,8.868384,10.933294,8.463276,10.375744,8.906716


,test_index,0,1,2,3,4,5,6,7,8,9,10
0,0,8.057694,6.369901,6.075346,6.102559,7.215975,7.985484,7.211557,6.171700,7.085064,6.973543,6.133398
1,1,8.791942,7.104965,6.839477,6.850126,7.839132,8.123855,8.037867,7.316548,8.025189,7.621195,6.909753
2,2,8.693665,7.116394,7.323831,6.745236,7.353082,7.437795,7.982075,7.331060,8.259459,7.693481,6.885509
3,3,8.153925,6.669498,6.342122,6.249975,7.026427,7.989561,7.463363,6.742881,6.953684,7.115582,6.284134
4,4,8.905309,7.479300,7.771489,7.297091,7.843849,8.142936,8.320691,7.748460,8.370316,8.234035,7.429521
